In [1]:
import json
import os
import re
import time
from datetime import datetime

import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
ollama_url = os.getenv("OLLAMA_URL")

print("OpenAI key :", "있음" if openai_api_key else "없음 — .env 를 확인하세요")
print("Ollama URL :", ollama_url)

try:
    probe = OpenAI(api_key="ollama", base_url=ollama_url)
    installed = sorted(m.id for m in probe.models.list())
    print("로컬 모델  :", ", ".join(installed) or "없음 — ollama pull 로 받으세요")
except Exception as e:
    print("로컬 모델  : 연결 실패 —", type(e).__name__)
    print("             Ollama 가 떠 있는지 터미널에서 'ollama list' 로 확인하세요")

OpenAI key : 있음
Ollama URL : http://localhost:11434/v1
로컬 모델  : llama3.2:3b, llama3.2:latest, qwen2.5:3b


In [2]:
advisories = json.loads(open("advisories_sample.json", encoding="utf-8").read())

print(f"공지 {len(advisories)}건\n")
for a in advisories:
    print(f"{a['no']:2}. {a['cve_id']:16} {a['label']:22} {len(a['advisory']):5}자")

공지 10건

 1. CVE-2021-44228   Apache Log4j2           2500자
 2. CVE-2022-22965   Spring Framework        1830자
 3. CVE-2017-5638    Apache Struts 2         1726자
 4. CVE-2014-6271    GNU Bash                1011자
 5. CVE-2023-49105   ownCloud                 383자
 6. CVE-2026-48710   Starlette               2500자
 7. CVE-2026-59822   LiteLLM                 1752자
 8. CVE-2026-75650   Adobe Commerce           350자
 9. CVE-2026-49869   Kestra                   772자
10. CVE-2026-73570   Zimbra Collaboration     418자


In [3]:
print(advisories[4]["cve_id"], "|", advisories[4]["label"])
print()
print(advisories[4]["advisory"])

CVE-2023-49105 | ownCloud

An issue was discovered in ownCloud owncloud/core before 10.13.1. An attacker can access, modify, or delete any file without authentication if the username of a victim is known, and the victim has no signing-key configured. This occurs because pre-signed URLs can be accepted even when no signing-key is configured for the owner of the files. The earliest affected version is 10.6.0.


In [4]:
SYSTEM_PROMPT = """당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.
- 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.
- 한국어로 답하세요."""


def build_messages(advisory_text):
    """공지 하나를 messages 로 만든다."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"다음 보안 공지를 요약하세요.\n\n{advisory_text}"},
    ]


print(SYSTEM_PROMPT)

당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.
- 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.
- 한국어로 답하세요.


In [5]:
openai_client = OpenAI()
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

MODELS = {
    "gpt-4.1-mini": (openai_client, "gpt-4.1-mini", 300),
    "gpt-4.1-nano": (openai_client, "gpt-4.1-nano", 300),
    "qwen2.5:3b":   (ollama_client, "qwen2.5:3b",   300),
    "llama3.2":     (ollama_client, "llama3.2",     300),
}

print("비교할 모델", len(MODELS), "개")
for name, (_, model_id, mt) in MODELS.items():
    where = "로컬" if _ is ollama_client else "클라우드"
    print(f"  {name:16} {where:5} -> {model_id:16} max_tokens={mt}")

비교할 모델 4 개
  gpt-4.1-mini     클라우드  -> gpt-4.1-mini     max_tokens=300
  gpt-4.1-nano     클라우드  -> gpt-4.1-nano     max_tokens=300
  qwen2.5:3b       로컬    -> qwen2.5:3b       max_tokens=300
  llama3.2         로컬    -> llama3.2         max_tokens=300


In [6]:
encoding = tiktoken.get_encoding("o200k_base")   # gpt-4.1 계열


def count_tokens(messages):
    return sum(len(encoding.encode(m["content"])) for m in messages)


total_in = 0
print(f"{'번호':>4} {'CVE':16} {'입력 토큰':>9}")
for a in advisories:
    n = count_tokens(build_messages(a["advisory"]))
    total_in += n
    print(f"{a['no']:>4} {a['cve_id']:16} {n:>9,}")

print(f"\n공지 10건 합계: {total_in:,} 토큰 (모델 1개 기준)")
print(f"모델 {len(MODELS)}개면 약 {total_in * len(MODELS):,} 토큰")

  번호 CVE                  입력 토큰
   1 CVE-2021-44228         789
   2 CVE-2022-22965         633
   3 CVE-2017-5638          573
   4 CVE-2014-6271          370
   5 CVE-2023-49105         274
   6 CVE-2026-48710         672
   7 CVE-2026-59822         560
   8 CVE-2026-75650         250
   9 CVE-2026-49869         350
  10 CVE-2026-73570         269

공지 10건 합계: 4,740 토큰 (모델 1개 기준)
모델 4개면 약 18,960 토큰


In [7]:
# no: (제품 낱말, [조치 낱말 — 하나만 맞으면 정답])
ANSWER_KEY = {
    1:  ("Log4j",          ["2.16.0"]),
    2:  ("Spring",         ["5.3.18", "5.2.20"]),
    3:  ("Struts",         ["2.3.32", "2.5.10.1"]),
    4:  ("Bash",           []),            # 공지에 버전 없음
    5:  ("ownCloud",       ["10.13.1"]),
    6:  ("Starlette",      ["1.0.1"]),
    7:  ("LiteLLM",        ["1.84.0"]),
    8:  ("Adobe Commerce", []),            # 공지에 버전 없음
    9:  ("Kestra",         ["1.0.45", "1.3.21"]),
    10: ("Zimbra",         ["10.1.20"]),
}

REQUIRED_LINES = ["제품:", "영향:", "조치:", "긴급도:"]
NO_VERSION = "공지에 없음"


def has_line(answer, prefix):
    """그 줄이 '제품:' 으로 시작하는가.

    "제품:" 이 답변 어딘가에 있기만 하면 되는 걸로 검사하면 안 됩니다.
    어떤 모델은 "# 제품: ..." 처럼 앞에 기호를 붙입니다.
    그러면 검사는 통과하지만 아래 get_line() 은 그 줄을 못 꺼냅니다.
    검사와 파싱이 같은 기준을 써야 점수가 거짓말을 하지 않습니다.
    """
    return any(line.strip().startswith(prefix) for line in answer.splitlines())


def get_line(answer, prefix):
    """답변에서 '조치:' 같은 줄 하나를 꺼낸다."""
    for line in answer.splitlines():
        if line.strip().startswith(prefix):
            return line.split(":", 1)[1].strip()
    return ""


def grade(no, answer):
    """형식 · 제품 · 조치 세 가지를 채점한다."""
    product, versions = ANSWER_KEY[no]

    form = all(has_line(answer, p) for p in REQUIRED_LINES)
    prod = product.lower() in answer.lower()

    action = get_line(answer, "조치:")
    if versions:
        act = any(v in action for v in versions)
        made_up = False
    else:
        # 버전이 없는 공지 — "공지에 없음" 이라고 해야 정답
        # 공백을 양쪽 다 지우고 비교합니다. "공지에없음" 처럼 붙여 쓰는 모델이 있습니다.
        act = NO_VERSION.replace(" ", "") in action.replace(" ", "")
        # 숫자가 들어간 버전처럼 보이는 말을 했다면 지어낸 것
        made_up = bool(re.search(r"\d+\.\d+", action))

    return {"형식": form, "제품": prod, "조치": act, "지어냄": made_up}


print("채점표", len(ANSWER_KEY), "건")
print("버전이 공지에 없는 건:", [n for n, (_, v) in ANSWER_KEY.items() if not v])

채점표 10 건
버전이 공지에 없는 건: [4, 8]


In [8]:
def ask(client, model_id, advisory_text, max_tokens):
    """공지 하나를 요약시킨다. 결과와 메타데이터를 함께 돌려준다.

    클라우드인지 로컬인지 이 함수는 모릅니다. client 만 다를 뿐입니다.
    """
    started = time.time()
    response = client.chat.completions.create(
        model=model_id,
        messages=build_messages(advisory_text),
        temperature=0,        # 같은 입력에 같은 답 — 비교의 전제
        max_tokens=max_tokens,
    )
    choice = response.choices[0]
    usage = response.usage
    return {
        "answer": choice.message.content or "",
        "finish_reason": choice.finish_reason,
        "in_tokens": usage.prompt_tokens,
        "out_tokens": usage.completion_tokens,
        "seconds": round(time.time() - started, 1),
        "error": None,
    }


results = {}      # {모델이름: {공지번호: 결과}}

for name, (client, model_id, max_tokens) in MODELS.items():
    print(f"===== {name} =====")
    results[name] = {}
    for a in advisories:
        try:
            r = ask(client, model_id, a["advisory"], max_tokens)
        except Exception as e:
            r = {"answer": "", "finish_reason": None, "in_tokens": 0, "out_tokens": 0,
                 "seconds": 0, "error": f"{type(e).__name__}: {e}"}
            print(f"  {a['no']:2}. {a['cve_id']:16} [오류] {r['error'][:60]}")
        else:
            g = grade(a["no"], r["answer"])
            mark = "".join("O" if g[k] else "X" for k in ("형식", "제품", "조치"))
            warn = "  <- 지어냄" if g["지어냄"] else ""
            print(f"  {a['no']:2}. {a['cve_id']:16} {mark}  "
                  f"{r['finish_reason']:6} {r['seconds']:5.1f}초{warn}")
        results[name][a["no"]] = r
    print()

print("완료")

===== gpt-4.1-mini =====
   1. CVE-2021-44228   OOO  stop     1.5초
   2. CVE-2022-22965   OOO  stop     1.0초
   3. CVE-2017-5638    OOO  stop     1.2초
   4. CVE-2014-6271    OOO  stop     0.8초
   5. CVE-2023-49105   OOO  stop     0.9초
   6. CVE-2026-48710   OOO  stop     1.0초
   7. CVE-2026-59822   OOO  stop     0.9초
   8. CVE-2026-75650   OOO  stop     1.1초
   9. CVE-2026-49869   OOO  stop     1.0초
  10. CVE-2026-73570   OOO  stop     1.3초

===== gpt-4.1-nano =====
   1. CVE-2021-44228   OOX  stop     0.8초
   2. CVE-2022-22965   OOO  stop     1.2초
   3. CVE-2017-5638    OOX  stop     1.0초
   4. CVE-2014-6271    OOO  stop     1.0초
   5. CVE-2023-49105   OOX  stop     1.0초
   6. CVE-2026-48710   OOO  stop     0.9초
   7. CVE-2026-59822   OOO  stop     1.0초
   8. CVE-2026-75650   OOO  stop     0.9초
   9. CVE-2026-49869   OOO  stop     1.1초
  10. CVE-2026-73570   OOX  stop     0.8초

===== qwen2.5:3b =====
   1. CVE-2021-44228   OOO  stop    19.9초
   2. CVE-2022-22965   OOO  stop    16.6초
 

In [9]:
def summarize(name):
    rows = results[name]
    s = {"형식": 0, "제품": 0, "조치": 0, "지어냄": 0,
         "잘림": 0, "오류": 0, "입력": 0, "출력": 0, "초": 0.0}
    for a in advisories:
        r = rows[a["no"]]
        if r["error"]:
            s["오류"] += 1
            continue
        g = grade(a["no"], r["answer"])
        for k in ("형식", "제품", "조치", "지어냄"):
            s[k] += int(g[k])
        if r["finish_reason"] == "length":
            s["잘림"] += 1
        s["입력"] += r["in_tokens"]
        s["출력"] += r["out_tokens"]
        s["초"] += r["seconds"]
    return s


scores = {name: summarize(name) for name in MODELS}

n = len(advisories)
header = (f"{'모델':20}{'형식':>6}{'제품':>6}{'조치':>6}{'지어냄':>7}"
          f"{'잘림':>6}{'오류':>6}{'입력토큰':>10}{'출력토큰':>10}{'총초':>7}")
print(header)
print("-" * len(header))
for name, s in scores.items():
    print(f"{name:20}{s['형식']:>4}/{n}{s['제품']:>4}/{n}{s['조치']:>4}/{n}"
          f"{s['지어냄']:>7}{s['잘림']:>6}{s['오류']:>6}"
          f"{s['입력']:>10,}{s['출력']:>10,}{s['초']:>7.1f}")

모델                      형식    제품    조치    지어냄    잘림    오류      입력토큰      출력토큰     총초
------------------------------------------------------------------------------------
gpt-4.1-mini          10/10  10/10  10/10      0     0     0     4,850       670   10.7
gpt-4.1-nano          10/10  10/10   6/10      0     0     0     4,850       550    9.7
qwen2.5:3b            10/10  10/10   8/10      0     0     0     5,088       953   86.8
llama3.2               9/10  10/10   4/10      1     0     0     5,015       649   73.8


In [10]:
for no in (4, 8):
    label = next(a["label"] for a in advisories if a["no"] == no)
    print("=" * 72)
    print(f"{no}번 — {label}  (공지에 패치 버전 없음)")
    print("=" * 72)
    for name in MODELS:
        r = results[name][no]
        if r["error"]:
            print(f"  {name:20} [오류]")
            continue
        action = get_line(r["answer"], "조치:")
        g = grade(no, r["answer"])
        verdict = "지어냄 !!" if g["지어냄"] else ("정답" if g["조치"] else "애매")
        print(f"  {name:20} [{verdict:9}] {action[:70]}")
    print()

4번 — GNU Bash  (공지에 패치 버전 없음)
  gpt-4.1-mini         [정답       ] 공지에 없음
  gpt-4.1-nano         [정답       ] 공지에 없음
  qwen2.5:3b           [정답       ] 공지에 없음
  llama3.2             [지어냄 !!   ] GNU Bash 4.4 이상

8번 — Adobe Commerce  (공지에 패치 버전 없음)
  gpt-4.1-mini         [정답       ] 공지에 없음
  gpt-4.1-nano         [정답       ] 공지에 없음
  qwen2.5:3b           [정답       ] 공지에 없음
  llama3.2             [정답       ] 공지에 없음



In [11]:
# 답변 전문을 하나만 펼쳐 봅니다
sample_model = list(MODELS)[0]
print(f"--- {sample_model} / 8번 Adobe Commerce 전문 ---")
print(results[sample_model][8]["answer"])

--- gpt-4.1-mini / 8번 Adobe Commerce 전문 ---
제품: Adobe Commerce (영향받는 버전 공지에 없음)  
영향: 공격자가 임의의 코드를 실행할 수 있음  
조치: 공지에 없음  
긴급도: 높음


In [13]:
SYSTEM_NO_GUARD = """당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전>
긴급도: <긴급|높음|보통|낮음 중 하나>

한국어로 답하세요."""


def ask_with_system(system_prompt, advisory_text, max_tokens=300):
    """system 만 바꿔서 같은 공지를 다시 물어본다."""
    messages = build_messages(advisory_text)
    messages[0] = {"role": "system", "content": system_prompt}   # system 만 교체
    r = openai_client.chat.completions.create(
        model="gpt-4.1-mini", temperature=0, max_tokens=max_tokens, messages=messages)
    return r.choices[0].message.content or ""


# 한 번만 돌려 보고 결론 내지 않습니다. temperature=0 이어도 말이 조금씩 달라집니다.
for no in (4, 8):
    a = next(x for x in advisories if x["no"] == no)
    print("=" * 72)
    print(f"{no}번 — {a['label']}  (공지에 패치 버전 없음)")
    for tag, sp in [("지시 있음", SYSTEM_PROMPT), ("지시 없음", SYSTEM_NO_GUARD)]:
        for trial in range(3):
            action = get_line(ask_with_system(sp, a["advisory"]), "조치:")
            made_up = bool(re.search(r"\d+\.\d+", action))
            flag = "지어냄 !!" if made_up else "안 지어냄 "
            print(f"  [{tag}] {trial + 1}회  {flag}  조치: {action[:60]}")
    print()

4번 — GNU Bash  (공지에 패치 버전 없음)
  [지시 있음] 1회  안 지어냄   조치: 공지에 없음
  [지시 있음] 2회  안 지어냄   조치: 공지에 없음
  [지시 있음] 3회  안 지어냄   조치: 공지에 없음
  [지시 없음] 1회  지어냄 !!  조치: 4.3 이후 버전으로 업데이트
  [지시 없음] 2회  지어냄 !!  조치: GNU Bash 4.3 이상으로 업데이트
  [지시 없음] 3회  지어냄 !!  조치: 4.3 이후 버전으로 업데이트

8번 — Adobe Commerce  (공지에 패치 버전 없음)
  [지시 있음] 1회  안 지어냄   조치: 공지에 없음
  [지시 있음] 2회  안 지어냄   조치: 공지에 없음
  [지시 있음] 3회  안 지어냄   조치: 공지에 없음
  [지시 없음] 1회  안 지어냄   조치: 최신 버전으로 업데이트 권장
  [지시 없음] 2회  안 지어냄   조치: 최신 보안 패치 적용 권장
  [지시 없음] 3회  안 지어냄   조치: 최신 버전으로 업데이트 권장



In [14]:
def save_markdown(filepath="exercise_result.md"):
    """점수표와 답변 전문을 markdown 으로 저장한다.

    results 는 Step 7, scores 는 Step 8 에서 만들어집니다.
    """
    for need, step in [("results", "Step 7 (전체 실행)"), ("scores", "Step 8 (점수표)")]:
        if need not in globals():
            raise RuntimeError(f"{need} 가 없습니다. {step} 셀을 먼저 실행하세요.")

    lines = [
        "# 보안 공지 요약 — 모델 비교 결과",
        "",
        f"_실행: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        f"_공지 {len(advisories)}건 × 모델 {len(MODELS)}개 = {len(advisories) * len(MODELS)}회 호출_",
        "",
        "## 점수표",
        "",
        "| 모델 | 형식 | 제품 | 조치 | 지어냄 | 잘림 | 오류 | 입력토큰 | 출력토큰 | 총 초 |",
        "|---|---|---|---|---|---|---|---|---|---|",
    ]
    n = len(advisories)
    for name, s in scores.items():
        lines.append(
            f"| {name} | {s['형식']}/{n} | {s['제품']}/{n} | {s['조치']}/{n} | "
            f"**{s['지어냄']}** | {s['잘림']} | {s['오류']} | "
            f"{s['입력']:,} | {s['출력']:,} | {s['초']:.1f} |")

    lines += ["", "## 버전이 공지에 없는 두 건", ""]
    for no in (4, 8):
        label = next(a["label"] for a in advisories if a["no"] == no)
        lines += [f"### {no}번 — {label}", ""]
        for name in MODELS:
            r = results[name][no]
            action = "[오류]" if r["error"] else get_line(r["answer"], "조치:")
            lines.append(f"- **{name}**: {action}")
        lines.append("")

    lines += ["## 모델별 전체 답변", ""]
    for name in MODELS:
        lines += [f"### {name}", ""]
        for a in advisories:
            r = results[name][a["no"]]
            lines += [f"**{a['no']}. {a['cve_id']} ({a['label']})**", ""]
            lines += ["```", r["error"] or r["answer"], "```", ""]

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"저장: {filepath}  ({len(chr(10).join(lines)):,}자)")
    return filepath


save_markdown()

저장: exercise_result.md  (8,175자)


'exercise_result.md'

In [16]:
for need, step in [("results", "Step 7 (전체 실행)"), ("scores", "Step 8 (점수표)")]:
    if need not in globals():
        raise RuntimeError(f"{need} 가 없습니다. {step} 셀을 먼저 실행하세요.")

In [17]:
n = len(advisories)
print("정리\n")

# 호출이 전부 실패한 모델은 후보에서 뺍니다.
# 답이 하나도 없으면 '지어냄 0' 이 저절로 참이 되어 버리기 때문입니다.
answered = [name for name, s in scores.items() if s["오류"] < n]
clean = [name for name in answered if scores[name]["지어냄"] == 0]
print("답을 받은 모델    :", ", ".join(answered) or "없음 !!")
print("지어내지 않은 모델:", ", ".join(clean) or "없음 !!")

# 정확도 = 제품 + 조치
ranked = sorted(answered, key=lambda m: -(scores[m]["제품"] + scores[m]["조치"]))
print(f"\n정확도 순 (제품 + 조치, 만점 {n * 2}):")
for m in ranked:
    s = scores[m]
    print(f"  {m:16} {s['제품'] + s['조치']:2}/{n * 2}  "
          f"(제품 {s['제품']}/{n}, 조치 {s['조치']}/{n})  "
          f"토큰 {s['입력'] + s['출력']:>6,}  {s['초']:5.0f}초")

print("\n'지어냄 0' 만 보고 고르지 마세요.")
print("모든 공지에 \"공지에 없음\" 이라고만 답해도 지어냄은 0 입니다.")
print("조치 점수를 반드시 같이 보세요.")

정리

답을 받은 모델    : gpt-4.1-mini, gpt-4.1-nano, qwen2.5:3b, llama3.2
지어내지 않은 모델: gpt-4.1-mini, gpt-4.1-nano, qwen2.5:3b

정확도 순 (제품 + 조치, 만점 20):
  gpt-4.1-mini     20/20  (제품 10/10, 조치 10/10)  토큰  5,520     11초
  qwen2.5:3b       18/20  (제품 10/10, 조치 8/10)  토큰  6,041     87초
  gpt-4.1-nano     16/20  (제품 10/10, 조치 6/10)  토큰  5,400     10초
  llama3.2         14/20  (제품 10/10, 조치 4/10)  토큰  5,664     74초

'지어냄 0' 만 보고 고르지 마세요.
모든 공지에 "공지에 없음" 이라고만 답해도 지어냄은 0 입니다.
조치 점수를 반드시 같이 보세요.
